# Two-relation replication on fresh seeds (10-19)

`ablation_stage_a.ipynb` ran 22 relation arms over split seeds 1-5 and found exactly two relations that
matter, both of them the ones `DistMultPredictor`'s disease signature reads:

```
drop_disease_protein_full   d_indication -0.1042   d_contraindication -0.1483   rank -0.1263
drop_disease_disease_full   d_indication -0.0533   d_contraindication -0.0194   rank -0.0363
drop_drug_protein           d_indication -0.0135   d_contraindication -0.0058   rank -0.0096   <- 3rd
...every other relation      |rank| <= 0.006
```

This notebook re-runs **only those two arms**, plus the baseline they are measured against, on **ten
fresh split seeds (10-19)**. Nothing about the ablation changes - it is the same graph-and-signature
(`_full`) ablation, the same model, the same schedule, the same metric. The only purpose is to turn an
n=5 result into an n=10 result on seeds that did not inform the choice of which arms to run.

That last point is what makes it worth the GPU time: seeds 1-5 both discovered *and* estimated the
effect, so their effect size is optimistically biased by selection over 22 arms. Seeds 10-19 estimate it
on data that played no part in picking the winners.

### Arms (3, x 10 seeds = 30 runs)

| arm | graph | signature `disease_etypes` |
|---|---|---|
| `baseline` | full, 50 edge types | `['disease_disease', 'rev_disease_protein']` |
| `drop_disease_protein_full` | `disease_protein` + `rev_disease_protein` removed | `['disease_disease']` |
| `drop_disease_disease_full` | `disease_disease` removed | `['rev_disease_protein']` |

`baseline` has to be re-run here: Stage A's baseline is seeds 1-5, and a paired delta needs the same
seed on both sides. Where a sibling grid already has a configuration-identical baseline for one of these
seeds it is imported rather than retrained (section 5).

### Results and resumability

Written to `results/ablation_stage_a_seeds10_19/<run_key>/` the moment each run finishes; a run with both
output files is skipped. Safe to re-run from cell 1 after a crash, kernel restart or interrupt - which is
what `run_ablation_*.sh`-style papermill restart loops rely on.

---
## 1 - Environment

In [ ]:
import os, sys, json, time, gc, random, shutil, inspect, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')


def _find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'pyg_implementation' / 'txgnn').is_dir():
            return cand
    raise RuntimeError('Could not find pyg_implementation/txgnn above ' + str(p))


REPO_ROOT = _find_repo_root()
PYG_ROOT  = REPO_ROOT / 'pyg_implementation'

# Put the PyG port FIRST on sys.path so `import txgnn` never resolves to dgl_implementation/.
sys.path.insert(0, str(PYG_ROOT))

import torch
import txgnn
import txgnn.model as txgnn_model
from txgnn import TxData, TxGNN
from txgnn.utils import evaluate_fb, create_pyg_graph, obtain_disease_profile

assert 'pyg_implementation' in txgnn.__file__, (
    'Wrong txgnn package on sys.path (%s). Restart the kernel.' % txgnn.__file__)

# Guard for requirement 1: the Gumbel-gate switch was removed from the library in commit 2d064cc.
# Passing it (or reading m.config['use_decision_network'], or asserting on m.model.pred.block_mode)
# the way the old notebook's cell 43 did is a TypeError / KeyError now. Fail here, loudly, rather
# than eight hours into the grid.
_MI_PARAMS = set(inspect.signature(TxGNN.model_initialize).parameters)
assert 'use_decision_network' not in _MI_PARAMS, (
    'model_initialize accepts use_decision_network again - revisit MODEL_CFG before trusting this grid.')


def set_all_seeds(seed: int):
    '''Seed every RNG the training path touches.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

print('repo root     :', REPO_ROOT)
print('txgnn package :', txgnn.__file__)
print('torch         :', torch.__version__)
print('device        :', DEVICE, '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor())
print('model_initialize params:', sorted(_MI_PARAMS - {'self'}))
if DEVICE == 'cpu':
    print('\nWARNING: CPU only. A ~45 GPU-hour grid is not practical here.')

---
## 2 - Configuration

Model and schedule are copied from the `full` profile of `TxGNN_PyG_Train_Eval.ipynb` (cell 8), which in
turn mirrors `pyg_implementation/reproduce/train.py`.

**Pretraining is one epoch.** `PRETRAIN_CFG['n_epoch'] = 1` - that one epoch is the full ~6,578-step
schedule over every training edge and is what costs ~19 min; it is not one epoch out of many. The
`smoke` profile's `n_epoch=0` is the only other value the original notebook ever uses.

The single deliberate deviation is `train_print_per_n`, raised from 100/5 to 500/100. It controls console
logging only (`TxGNN.py:191` and `TxGNN.py:253` gate a `print` and a `get_all_metrics_fb` call that
consumes no RNG and feeds nothing back into training), and 30 runs at the original setting would emit
~6,000 metric blocks. Everything that can change a number - `n_epoch`, `learning_rate`, `batch_size`,
`valid_per_n` - is untouched.

In [ ]:
SPLIT   = 'complex_disease'
SEEDS   = list(range(10, 20))      # ten fresh split seeds: 10, 11, ... 19
PROFILE = 'full'

# verbatim from the 'full' profile / reproduce/train.py - no use_decision_network (see cell above)
MODEL_CFG = dict(
    n_hid       = 100,
    n_inp       = 100,
    n_out       = 100,
    proto       = True,                  # disease-similarity metric learning
    proto_num   = 3,
    attention   = False,
    sim_measure = 'all_nodes_profile',   # signature = binary vector over disease + gene/protein neighbours
    agg_measure = 'rarity',
)

PRETRAIN_CFG = dict(n_epoch=1,    learning_rate=1e-3, batch_size=1024, train_print_per_n=500)
FINETUNE_CFG = dict(n_epoch=1000, learning_rate=5e-4, train_print_per_n=100, valid_per_n=20)

DD_REL     = ['contraindication', 'indication', 'off-label use']
DD_REV_REL = ['rev_' + r for r in DD_REL]
DD_ETYPES  = [('drug', r, 'disease') for r in DD_REL] + [('disease', r, 'drug') for r in DD_REV_REL]

DATA_FOLDER = REPO_ROOT / 'data'
ABL_ROOT    = REPO_ROOT / 'results' / 'ablation_stage_a_seeds10_19'
ABL_ROOT.mkdir(parents=True, exist_ok=True)

# Stage A's own results, for the replication comparison in section 7.1
STAGE_A_ROOT = REPO_ROOT / 'results' / 'ablation_stage_a'

# sibling grids that may already hold a configuration-identical baseline for one of these seeds
BASELINE_SOURCES = [
    (STAGE_A_ROOT, 'baseline'),
    (REPO_ROOT / 'results' / 'ablation_signature_only', 'baseline'),
    (REPO_ROOT / 'results' / 'ablation_encoder_swap',   'rgcn'),
]

# --- switches ------------------------------------------------------------------------------------
RUN_GRID          = True   # section 6.2 trains; set False to inspect the plan / aggregate only
SAVE_MODELS       = False
IMPORT_BASELINE   = True   # reuse a sibling grid's baseline for a shared seed - see section 5
STOP_AFTER_N_RUNS = None   # e.g. 2 for a short unattended smoke pass over the harness

print('split   :', SPLIT)
print('seeds   :', SEEDS)
print('results :', ABL_ROOT)
print('pretrain:', PRETRAIN_CFG)
print('finetune:', FINETUNE_CFG)

---
## 3 - The arms

Only the two relations Stage A identified, ablated exactly as Stage A ablated them: removed from
`df_train` / `df_valid` / `df_test` **and** from the signature `disease_etypes`
(`model.py:88`, fallback at `model.py:206`). That is what the `_full` suffix means, and the arm names are
kept byte-identical to Stage A's so the two grids can be compared without a lookup table.

`disease_protein` is a hetero relation, so dropping it removes `disease_protein` and
`rev_disease_protein`. `disease_disease` is same-type, so both directions already live under the one
relation name (`utils.reverse_rel_generation` only renames when `x_type != y_type`) and the single name
covers them. The drop helper filters on `{rel, 'rev_' + rel}`, which is correct either way.

Not included, and deliberately so: an arm dropping **both** relations at once. It would leave the
signature with no source at all, and the empty-signature path is a confounded ablation rather than a
clean one - `sim` goes to zero, the pooled prototype goes to zero, but the `rarity` gate still scales the
disease embedding by `(1 - c_i)` with `c_i ~ 0.9` on this split. `ablation_signature_only.ipynb` covers
that case (`sig_drop_both`) and documents the caveat.

In [ ]:
# Stage A's two winners, named exactly as Stage A named them.
ABLATE_RELATIONS = ['disease_protein', 'disease_disease']

# relation -> entries to strip from DistMultPredictor's disease_etypes (model.py:88, model.py:206).
# Note the asymmetry: dropping the *relation* 'disease_protein' removes the *signature entry*
# 'rev_disease_protein', because the signature reads the reverse direction (disease -> protein).
SIGNATURE_RELATIONS = {
    'disease_protein': ['rev_disease_protein'],
    'disease_disease': ['disease_disease'],
}
assert set(ABLATE_RELATIONS) == set(SIGNATURE_RELATIONS), \
    'both arms here are signature relations; a graph-only arm belongs in ablation_stage_a.ipynb'


def _make_arm(name, drop, sig_drop=()):
    return {'arm': name, 'drop': list(drop), 'sig_drop': list(sig_drop),
            'sig_ablated': bool(sig_drop)}


ARMS = [_make_arm('baseline', [])]
for rel in ABLATE_RELATIONS:
    # '_full' = graph + signature, matching Stage A's run_key so the grids line up
    ARMS.append(_make_arm('drop_%s_full' % rel, [rel], SIGNATURE_RELATIONS[rel]))

ARMS_BY_NAME = {a['arm']: a for a in ARMS}
assert len(ARMS_BY_NAME) == len(ARMS), 'duplicate arm name'

print('%d arms (1 baseline + %d ablations) x %d seeds = %d runs\n'
      % (len(ARMS), len(ARMS) - 1, len(SEEDS), len(ARMS) * len(SEEDS)))
for a in ARMS:
    print('  %-28s drop=%-24s signature: %s'
          % (a['arm'], a['drop'] or '-',
             '-' + ','.join(a['sig_drop']) if a['sig_drop'] else 'unchanged'))

---
## 4 - Ablation mechanics

Three pieces:

1. `drop_relations(df, rels)` - filter on `{rel, 'rev_' + rel}`.
2. `AblatedData` - a stand-in for `TxData` holding the filtered frames and a rebuilt graph. `TxGNN`
   only reads `.G`, `.df`, `.df_train`, `.df_valid`, `.df_test`, `.data_folder`, `.disease_eval_idx`,
   `.split`, `.no_kg`. The full `df` is shared by reference (`create_pyg_graph` only reads it), so the
   per-seed cache is never mutated.
3. A patch on `txgnn.model.obtain_disease_profile` that drops the ablated entries from the
   `(disease_etypes, disease_nodes)` pair before building the signature. Both call sites -
   `model.py:104` in the constructor and `model.py:214` in the unseen-disease fallback - resolve the
   name from the `txgnn.model` module globals at call time, so patching the module attribute covers
   both without editing the library.

Filtering `df_valid` / `df_test` is belt-and-braces: `evaluate_graph_construct` (`utils.py:549`) iterates
`G.edge_types`, so a relation already absent from the rebuilt graph could not enter the eval graphs
anyway. Doing it keeps the three frames consistent with each other and with what gets logged.

**Why the signature patch is semantically, not numerically, load-bearing.** Once the relation is gone
from `G`, `obtain_disease_profile` already returns an all-zero block for it (`utils.py:1020-1032`: no
matching edge type -> empty `nodes` -> zeros). `sim_matrix` is cosine, and appending a common zero block
to every vector changes neither the dot products nor the norms - so the similarity matrix is identical
either way. The patch makes the intent explicit, keeps the metadata honest, and halves the signature
tensor for these two arms; it does not change the numbers. Worth knowing before designing any follow-up
that assumes "graph-only" and "graph+signature" are two distinguishable variants here - they are not.

**Edge case, not reached by this grid.** If `disease_protein` and `disease_disease` were both dropped in
one run, `keep` would be empty and the signature would have no source at all. The patch then falls back
to the unpatched call, which returns an all-zero vector for every disease; `sim_matrix`'s `eps` floor
keeps that from becoming NaN, and every disease ends up equidistant, so prototype pooling degenerates to
an unweighted mean over `proto_num` arbitrary diseases. That is a documented degenerate mode, not a
supported configuration - the two relations are separate arms here, so it never happens.

In [ ]:
def drop_relations(df, rels):
    '''Remove `rels` and their rev_ twins from a split frame.'''
    victims = set()
    for r in rels:
        victims |= {r, 'rev_' + r}
    return df[~df.relation.isin(victims)].reset_index(drop=True)


class AblatedData:
    '''Minimal TxData stand-in carrying ablated splits and a rebuilt graph.'''

    def __init__(self, base, rels, data_folder, split):
        self.df = base['df']                       # untouched: fixes node counts across arms
        self.df_train = drop_relations(base['df_train'], rels)
        self.df_valid = drop_relations(base['df_valid'], rels)
        self.df_test  = drop_relations(base['df_test'],  rels)
        self.G = create_pyg_graph(self.df_train, self.df)
        self.data_folder = str(data_folder)
        self.split = split
        self.disease_eval_idx = None
        self.no_kg = False


# --- signature patch ------------------------------------------------------------------------------
SIG_DROP = set()          # set per run by run_one(); empty for every graph-only arm
_orig_obtain_disease_profile = obtain_disease_profile


def _obtain_disease_profile_ablated(G, disease, disease_etypes, disease_nodes):
    keep = [(e, n) for e, n in zip(disease_etypes, disease_nodes) if e not in SIG_DROP]
    if not keep:
        # degenerate mode - see the markdown above; unreachable in this grid
        return _orig_obtain_disease_profile(G, disease, disease_etypes, disease_nodes)
    etypes, nodes = zip(*keep)
    return _orig_obtain_disease_profile(G, disease, list(etypes), list(nodes))


txgnn_model.obtain_disease_profile = _obtain_disease_profile_ablated
assert txgnn_model.obtain_disease_profile is _obtain_disease_profile_ablated


# --- per-seed split cache -------------------------------------------------------------------------
_SEED_CACHE = {}


def get_seed_data(seed):
    '''Load (and cache) the unablated split frames for one seed.

    Holds one seed at a time - `df` is millions of rows. prepare_split() caches to disk, so the
    first call per seed writes data/<split>_<seed>/ and later calls just read the CSVs.
    '''
    if seed in _SEED_CACHE:
        return _SEED_CACHE[seed]
    _SEED_CACHE.clear()
    gc.collect()
    print('[data] loading split seed %d ...' % seed)
    t0 = time.time()
    set_all_seeds(seed)
    d = TxData(data_folder_path=str(DATA_FOLDER))
    d.prepare_split(split=SPLIT, seed=seed, no_kg=False)
    base = {'df': d.df, 'df_train': d.df_train, 'df_valid': d.df_valid, 'df_test': d.df_test}
    del d
    gc.collect()
    _SEED_CACHE[seed] = base
    print('[data] seed %d ready in %.1f s | train edges %d'
          % (seed, time.time() - t0, len(base['df_train'])))
    return base


def run_dir(arm, seed):
    return ABL_ROOT / ('%s_seed%d' % (arm, seed))


# --- reusing a sibling grid's baseline ------------------------------------------------------------
MODEL_KEYS_CHECKED = sorted(MODEL_CFG)       # the keys this notebook actually sets


def import_matching_run(src, dst, expect, extra=None):
    '''Copy a finished run from another grid iff its metadata matches `expect`. Returns (ok, why).'''
    sm, se = src / 'run_meta.json', src / 'edge_level_test_metrics.csv'
    if not (sm.exists() and se.exists()):
        return False, 'no run at %s' % src.name
    meta = json.loads(sm.read_text())
    for k, want in expect.items():
        got = meta.get(k, '<missing>')
        if isinstance(want, dict):
            if not isinstance(got, dict):
                return False, '%s is not a dict' % k
            bad = {kk: (got.get(kk, '<missing>'), vv) for kk, vv in want.items()
                   if got.get(kk, '<missing>') != vv}
            if bad:
                return False, '%s mismatch %s' % (k, bad)
        elif got != want:
            return False, '%s mismatch: %r != %r' % (k, got, want)

    dst.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(se, dst / 'edge_level_test_metrics.csv')
    meta.update(extra or {})
    meta['imported_from'] = str(src.relative_to(REPO_ROOT))
    meta['run_key'] = dst.name
    meta['arm'] = dst.name.rsplit('_seed', 1)[0]
    with open(dst / 'run_meta.json', 'w') as f:
        json.dump(meta, f, indent=2, default=str)
    return True, 'imported from %s' % src.relative_to(REPO_ROOT)


def try_import_baseline(seed):
    '''Look through BASELINE_SOURCES for an unablated run on this seed with an identical config.'''
    if not IMPORT_BASELINE:
        return False, 'disabled'
    expect = {'split': SPLIT, 'seed': seed,
              'dropped_edges_total': 0,
              'signature_disease_etypes': ['disease_disease', 'rev_disease_protein'],
              'model_config': {k: MODEL_CFG[k] for k in MODEL_KEYS_CHECKED},
              'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG}
    whys = []
    for root, arm in BASELINE_SOURCES:
        ok, why = import_matching_run(
            root / ('%s_seed%d' % (arm, seed)), run_dir('baseline', seed), expect,
            extra={'sig_ablated': False, 'dropped_relations_requested': [],
                   'dropped_relations_in_graph': [], 'dropped_edge_counts': {}})
        if ok:
            return True, why
        whys.append('%s/%s_seed%d: %s' % (root.name, arm, seed, why))
    return False, ' | '.join(whys)


def is_done(arm, seed):
    d = run_dir(arm, seed)
    return (d / 'edge_level_test_metrics.csv').exists() and (d / 'run_meta.json').exists()

---
## 5 - Plan

Enumerates every `run_key`, marks which are already on disk, and projects GPU-hours from the timings in
`results/TxGNN_complex_disease_seed1_full/run_config.json`, scaled by the fraction of training edges each
arm keeps (pretraining is one pass over every training edge in batches of 1,024, `TxGNN.py:141-148`).

**Splits for seeds 10-19 do not exist yet.** The first `get_seed_data(seed)` for each of them runs
`create_split`, which writes `data/complex_disease_<seed>/{train,valid,test}.csv` - a few minutes per
seed, once, and cached from then on. `data/kg_directed.csv` is shared across seeds and is rebuilt from
`data/kg.csv` only if missing (10-30 minutes, once ever).

**Baseline import.** Before planning, each `baseline_seed<n>` that is missing is offered the matching run
from each entry in `BASELINE_SOURCES` in turn - Stage A's `baseline`, the signature-only grid's
`baseline`, the encoder-swap grid's `rgcn`. All three are the same configuration; a run is accepted only
if its recorded `split`, `seed`, `model_config` (the eight keys this notebook sets), `pretrain`,
`finetune`, `dropped_edges_total == 0` and `signature_disease_etypes` all match, and the copy keeps an
`imported_from` field so nothing is silently borrowed. Stage A ran seeds 1-5, so it will not contribute
here; the other two grids run seeds 1-10, so seed 10 may be free.

In [ ]:
# --- offer each missing baseline seed a sibling grid's run before planning anything ---------------
if IMPORT_BASELINE:
    for _s in SEEDS:
        if is_done('baseline', _s):
            continue
        _ok, _why = try_import_baseline(_s)
        print('[import] baseline_seed%d: %s' % (_s, _why if _ok else 'will train'))
        if not _ok:
            print('          %s' % _why)

BASE_PRETRAIN_S, BASE_FINETUNE_S = 1155.7, 316.5   # fallback: seed-1 full run
_ref = REPO_ROOT / 'results' / ('TxGNN_%s_seed1_%s' % (SPLIT, PROFILE)) / 'run_config.json'
if _ref.exists():
    _r = json.loads(_ref.read_text())
    BASE_PRETRAIN_S = _r.get('pretrain_seconds', BASE_PRETRAIN_S)
    BASE_FINETUNE_S = _r.get('finetune_seconds', BASE_FINETUNE_S)
    print('timings from %s' % _ref.relative_to(REPO_ROOT))
print('baseline: pretrain %.1f min + finetune %.1f min = %.1f min/run\n'
      % (BASE_PRETRAIN_S / 60, BASE_FINETUNE_S / 60, (BASE_PRETRAIN_S + BASE_FINETUNE_S) / 60))

_base0   = get_seed_data(SEEDS[0])
_train   = _base0['df_train']
_counts  = _train.relation.value_counts()
N_TRAIN_EDGES = len(_train)

# both relations must actually exist, or the arm would silently be a second baseline
_missing = sorted(r for r in ABLATE_RELATIONS if r not in set(_counts.index))
assert not _missing, 'relations absent from df_train (seed %d): %s' % (SEEDS[0], _missing)

rows, total_s = [], 0.0
for a in ARMS:
    victims = sorted({v for r in a['drop'] for v in (r, 'rev_' + r)} & set(_counts.index))
    dropped = int(_counts.reindex(victims).fillna(0).sum())
    frac    = (N_TRAIN_EDGES - dropped) / N_TRAIN_EDGES
    est_s   = (BASE_PRETRAIN_S + BASE_FINETUNE_S) * frac
    done    = [s for s in SEEDS if is_done(a['arm'], s)]
    todo    = [s for s in SEEDS if s not in done]
    total_s += est_s * len(todo)
    rows.append({'arm': a['arm'],
                 'relations dropped': len(victims),
                 'train edges dropped': dropped,
                 'pct of KG': 100.0 * dropped / N_TRAIN_EDGES,
                 'signature ablated': a['sig_ablated'],
                 'est min/run': est_s / 60,
                 'done': len(done), 'todo': len(todo)})

plan = pd.DataFrame(rows)
print('train edges (seed %d, full KG): %d\n' % (SEEDS[0], N_TRAIN_EDGES))
display(plan.style.format({'pct of KG': '{:.2f}', 'est min/run': '{:.1f}'}).hide(axis='index'))

RUN_KEYS = [(a['arm'], s) for s in SEEDS for a in ARMS]     # seed-major, matches the run order
pending  = [k for k in RUN_KEYS if not is_done(*k)]
print('\nrun_keys (%d total, %d done, %d pending):'
      % (len(RUN_KEYS), len(RUN_KEYS) - len(pending), len(pending)))
for arm, seed in RUN_KEYS:
    print('  [%s] %s_seed%d' % ('x' if is_done(arm, seed) else ' ', arm, seed))
print('\nestimated remaining: %.1f GPU-hours (%.1f days unattended)'
      % (total_s / 3600, total_s / 86400))
print('order: seed-major - after seed %d finishes all three arms have n=1 and are already paired.'
      % SEEDS[0])

---
## 6 - The grid

`run_one` trains a single `(arm, seed)` and writes two files into
`results/ablation_stage_a_seeds10_19/<run_key>/`
the moment it finishes:

- `edge_level_test_metrics.csv` - AUROC/AUPRC for the six drug-disease edge types plus three summaries.
- `run_meta.json` - the sanity-check record: relation dropped, edge counts, `sorted(G.edge_types)` after
  ablation, the signature `disease_etypes` actually used, timings.

`MICRO (drug-disease)` is computed by `evaluate_fb` over the six DD edge types only, so it is comparable
across arms. `MACRO (all relations)` is **not** - `get_all_metrics_fb(full_mode=True)` averages over
`G.edge_types`, and an ablated graph has fewer terms in that mean. `MACRO (drug-disease)` is computed
here over the six DD relations, and is the comparable macro.

In [ ]:
def run_one(arm_name, seed, verbose=True):
    '''Train one (arm, seed) and write its results. Returns the edge-level DataFrame.'''
    global SIG_DROP

    arm  = ARMS_BY_NAME[arm_name]
    rdir = run_dir(arm_name, seed)
    el_csv, meta_json = rdir / 'edge_level_test_metrics.csv', rdir / 'run_meta.json'

    if el_csv.exists() and meta_json.exists():
        if verbose:
            print('[skip] %s_seed%d -> cached' % (arm_name, seed))
        return pd.read_csv(el_csv)

    if arm_name == 'baseline':
        ok, why = try_import_baseline(seed)
        if ok:
            print('[skip] baseline_seed%d -> %s' % (seed, why))
            return pd.read_csv(el_csv)

    print('\n' + '#' * 78)
    print('# %s   seed %d   (dropping %d relation(s), signature ablated: %s)'
          % (arm_name, seed, len(arm['drop']), arm['sig_ablated']))
    print('#' * 78)

    base     = get_seed_data(seed)
    n_before = len(base['df_train'])
    victims  = sorted({v for r in arm['drop'] for v in (r, 'rev_' + r)}
                      & set(base['df_train'].relation.unique()))
    per_rel  = {r: int((base['df_train'].relation == r).sum()) for r in victims}

    set_all_seeds(seed)
    d = AblatedData(base, arm['drop'], DATA_FOLDER, SPLIT)
    etypes_after = sorted('%s|%s|%s' % et for et in d.G.edge_types)

    # requirement 5: prove the relation actually left the graph
    still_there = [r for r in victims if any(et[1] == r for et in d.G.edge_types)]
    assert not still_there, 'ablation failed, still in G: %s' % still_there

    # signature ablation - set before model_initialize, which is where the profiles are built
    SIG_DROP   = set(arm['sig_drop'])
    sig_etypes = [e for e in ['disease_disease', 'rev_disease_protein'] if e not in SIG_DROP]
    if arm['sig_ablated']:
        print('[signature] disease_etypes %s -> %s  (metric learning ablated alongside message passing)'
              % (['disease_disease', 'rev_disease_protein'], sig_etypes))

    print('[graph] train edges %d -> %d (-%d, %.2f%%) | edge types %d | dropped: %s'
          % (n_before, len(d.df_train), n_before - len(d.df_train),
             100.0 * (n_before - len(d.df_train)) / n_before, len(etypes_after), per_rel or 'none'))

    set_all_seeds(seed)
    m = TxGNN(data=d, weight_bias_track=False, exp_name='%s_seed%d' % (arm_name, seed), device=DEVICE)
    m.model_initialize(**MODEL_CFG)

    t0 = time.time()
    set_all_seeds(seed)
    m.pretrain(**PRETRAIN_CFG)
    pre_s = time.time() - t0

    t0 = time.time()
    set_all_seeds(seed)
    m.finetune(**FINETUNE_CFG)
    fin_s = time.time() - t0

    # ---- edge-level test metrics ----
    (auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), test_loss = \
        evaluate_fb(m.best_model, m.g_test_pos, m.g_test_neg, m.G, m.dd_etypes, DEVICE, mode='test')

    el = pd.DataFrame(
        [('%s --%s--> %s' % et, auroc_rel.get(et, np.nan), auprc_rel.get(et, np.nan))
         for et in DD_ETYPES],
        columns=['relation', 'AUROC', 'AUPRC'])
    el.loc[len(el)] = ['MICRO (drug-disease)', micro_auroc, micro_auprc]
    el.loc[len(el)] = ['MACRO (drug-disease)',
                       float(np.nanmean([auroc_rel.get(et, np.nan) for et in DD_ETYPES])),
                       float(np.nanmean([auprc_rel.get(et, np.nan) for et in DD_ETYPES]))]
    el.loc[len(el)] = ['MACRO (all relations, NOT comparable across arms)', macro_auroc, macro_auprc]

    rdir.mkdir(parents=True, exist_ok=True)
    el.to_csv(el_csv, index=False)

    meta = {
        'run_key': '%s_seed%d' % (arm_name, seed),
        'experiment': 'stage_a_seeds10_19',
        'arm': arm_name, 'seed': seed, 'split': SPLIT, 'profile': PROFILE,
        'dropped_relations_requested': arm['drop'],
        'dropped_relations_in_graph': victims,
        'dropped_edge_counts': per_rel,
        'dropped_edges_total': n_before - len(d.df_train),
        'train_edges_before': n_before,
        'train_edges_after': len(d.df_train),
        'n_edge_types_after': len(etypes_after),
        'edge_types_after': etypes_after,
        'signature_ablated': arm['sig_ablated'],
        'signature_disease_etypes': sig_etypes,
        'signature_disease_etypes_default': ['disease_disease', 'rev_disease_protein'],
        'model_config': m.config,
        'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG,
        'pretrain_seconds': pre_s, 'finetune_seconds': fin_s,
        'test_bce_loss': float(test_loss),
        'torch': torch.__version__,
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'finished_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    }
    # write the metadata last - is_done() requires both files, so a crash mid-write leaves the run
    # pending rather than half-recorded
    with open(meta_json, 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    if SAVE_MODELS:
        mdir = REPO_ROOT / 'saved_models' / 'ablation_stage_a' / meta['run_key']
        mdir.mkdir(parents=True, exist_ok=True)
        m.save_model(str(mdir))

    ind = auprc_rel.get(('drug', 'indication', 'disease'), np.nan)
    con = auprc_rel.get(('drug', 'contraindication', 'disease'), np.nan)
    print('[done] %s_seed%d | %.1f min | indication AUPRC %.4f | contraindication AUPRC %.4f -> %s'
          % (arm_name, seed, (pre_s + fin_s) / 60, ind, con, rdir.relative_to(REPO_ROOT)))

    # ---- free the GPU before the next run ----
    SIG_DROP = set()
    del m, d, auroc_rel, auprc_rel
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return el

In [ ]:
# Seed-major: all three arms at one seed before moving on, so a partial grid is always a set of
# complete paired comparisons rather than an unpaired baseline. Also means one split load per seed.
if RUN_GRID:
    todo = [(a['arm'], s) for s in SEEDS for a in ARMS if not is_done(a['arm'], s)]
    if STOP_AFTER_N_RUNS:
        todo = todo[:STOP_AFTER_N_RUNS]

    print('%d run(s) pending of %d total.' % (len(todo), len(ARMS) * len(SEEDS)))
    grid_t0 = time.time()
    for i, (arm_name, seed) in enumerate(todo, 1):
        print('\n>>> %d/%d  %s_seed%d  (elapsed %.1f h)'
              % (i, len(todo), arm_name, seed, (time.time() - grid_t0) / 3600))
        run_one(arm_name, seed)
    print('\nGrid finished %d run(s) in %.1f h.' % (len(todo), (time.time() - grid_t0) / 3600))
else:
    print('RUN_GRID is False - no training. Set it in section 2.')

---
## 7 - Aggregate

Reads whatever is on disk - safe to run against a partial grid. Two views per relation:

- **absolute**: mean AUPRC over seeds with a 95% normal CI.
- **paired delta**: `arm - baseline` computed *per seed* and then averaged. Seeds are shared splits, so
  pairing removes split-to-split variance. An arm whose paired CI excludes zero is a relation the model
  genuinely uses.

`rank_score` is the mean of the indication and contraindication paired deltas; sorted ascending, so the
most damaging relation to drop sits at the top. With only two arms the ranking is not the point - the
effect sizes and their CIs are.

In [ ]:
def _load_all():
    recs = []
    for meta_p in sorted(ABL_ROOT.glob('*/run_meta.json')):
        el_p = meta_p.parent / 'edge_level_test_metrics.csv'
        if not el_p.exists():
            continue
        meta = json.loads(meta_p.read_text())
        el = pd.read_csv(el_p).set_index('relation')
        for rel, row in el.iterrows():
            recs.append({'arm': meta['arm'], 'seed': meta['seed'], 'relation': rel,
                         'AUROC': row['AUROC'], 'AUPRC': row['AUPRC'],
                         'dropped_edges': meta['dropped_edges_total'],
                         'signature_ablated': meta['signature_ablated']})
    return pd.DataFrame(recs)


def _ci95(x):
    x = np.asarray(x, dtype=float)
    n = len(x)
    return 1.96 * x.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan


long = _load_all()
summary, wide, base = None, None, None

if long.empty:
    print('No runs on disk yet - run the grid cell in section 6 first.')
else:
    ROW = {'indication': 'drug --indication--> disease',
           'contraindication': 'drug --contraindication--> disease',
           'micro': 'MICRO (drug-disease)'}

    wide = (long[long.relation.isin(ROW.values())]
            .pivot_table(index=['arm', 'seed'], columns='relation', values='AUPRC')
            .rename(columns={v: k for k, v in ROW.items()}))

    meta_by_arm = long.groupby('arm')[['dropped_edges', 'signature_ablated']].first()
    if 'baseline' in wide.index.get_level_values('arm'):
        base = wide.xs('baseline', level='arm')
    else:
        print('[warn] no baseline runs yet - deltas will be NaN.')

    out = []
    for arm, g in wide.groupby(level='arm'):
        g = g.droplevel('arm')
        rec = {'arm': arm, 'n_seeds': len(g),
               'dropped_edges': int(meta_by_arm.loc[arm, 'dropped_edges']),
               'signature_ablated': bool(meta_by_arm.loc[arm, 'signature_ablated'])}
        for key in ['indication', 'contraindication', 'micro']:
            rec['%s_AUPRC' % key] = g[key].mean()
            rec['%s_CI' % key] = _ci95(g[key])
            if arm == 'baseline':
                rec['%s_delta' % key], rec['%s_delta_CI' % key] = 0.0, np.nan
                rec['%s_n_paired' % key] = len(g)
            elif base is not None:
                paired = (g[key] - base[key]).dropna()      # inner-joins on seed
                rec['%s_delta' % key] = paired.mean() if len(paired) else np.nan
                rec['%s_delta_CI' % key] = _ci95(paired) if len(paired) > 1 else np.nan
                rec['%s_n_paired' % key] = len(paired)
            else:
                rec['%s_delta' % key], rec['%s_delta_CI' % key] = np.nan, np.nan
                rec['%s_n_paired' % key] = 0
        rec['rank_score'] = np.nanmean([rec['indication_delta'], rec['contraindication_delta']])
        out.append(rec)

    summary = pd.DataFrame(out).sort_values('rank_score', na_position='last').reset_index(drop=True)
    summary.to_csv(ABL_ROOT / 'seeds10_19_summary.csv', index=False)

    cols = ['arm', 'n_seeds', 'dropped_edges', 'signature_ablated',
            'indication_AUPRC', 'indication_CI', 'indication_delta', 'indication_delta_CI',
            'contraindication_AUPRC', 'contraindication_CI', 'contraindication_delta',
            'contraindication_delta_CI', 'micro_AUPRC', 'micro_delta', 'rank_score']
    print('%d arms, %d runs on disk -> %s\n'
          % (len(summary), len(wide), (ABL_ROOT / 'seeds10_19_summary.csv').relative_to(REPO_ROOT)))
    display(summary[cols].style.format('{:.4f}', subset=cols[4:], na_rep='-').hide(axis='index'))

    incomplete = summary[summary.n_seeds < len(SEEDS)]
    if len(incomplete):
        print('\n[partial] %d arm(s) below %d seeds - deltas are provisional: %s'
              % (len(incomplete), len(SEEDS), ', '.join(incomplete.arm)))

In [ ]:
# Per-seed paired deltas - the sign-consistency check a mean hides. With 10 seeds, 'wins' is the
# number of seeds on which dropping the relation actually hurt; 10/10 is a much stronger statement
# than a mean with a wide CI.
if summary is not None and base is not None:
    top = [a for a in summary.arm if a != 'baseline']
    for key in ['indication', 'contraindication']:
        tbl = pd.DataFrame({a: (wide.xs(a, level='arm')[key] - base[key]) for a in top}).T
        seed_cols = list(tbl.columns)
        tbl.columns = ['seed %d' % s for s in seed_cols]
        tbl['mean'] = tbl.mean(axis=1)
        tbl['wins'] = (tbl[['seed %d' % s for s in seed_cols]] < 0).sum(axis=1)
        print('\npaired delta AUPRC vs baseline - %s  (negative = dropping the relation hurt)' % key)
        display(tbl.style.format('{:.4f}', subset=[c for c in tbl.columns if c != 'wins'], na_rep='-'))

### 7.1 - Replication: seeds 1-5 (Stage A) vs seeds 10-19 (here)

The same two arms, the same ablation, two disjoint seed sets. Stage A's seeds both selected these arms
out of 22 and estimated their effect, so its effect size carries selection bias; this grid's does not.

`shrinkage` is `this / stage A` - below 1 means the effect is smaller on fresh seeds, which is what
selection bias predicts. A value near 1 means the Stage A estimate held up. The two estimates are
independent, so if their CIs overlap they agree.

In [ ]:
sa_csv = STAGE_A_ROOT / 'stage_a_summary.csv'
if summary is None:
    print('No local runs yet.')
elif not sa_csv.exists():
    print('Stage A summary not found at %s - run that notebook\'s aggregate cell.' % sa_csv)
else:
    sa = pd.read_csv(sa_csv).set_index('arm')
    sa_seeds = sorted(json.loads(p.read_text())['seed']
                      for p in STAGE_A_ROOT.glob('baseline_seed*/run_meta.json'))
    here = summary.set_index('arm')
    print('Stage A seeds: %s   |   this grid seeds: %s\n' % (sa_seeds, SEEDS))

    rows = []
    for arm in [a['arm'] for a in ARMS if a['arm'] != 'baseline']:
        if arm not in here.index or arm not in sa.index:
            continue
        for key in ['indication', 'contraindication', 'micro']:
            d_a, d_b = sa.loc[arm, '%s_delta' % key], here.loc[arm, '%s_delta' % key]
            rows.append({
                'arm': arm, 'metric': key,
                'stage A delta': d_a,
                'stage A CI': sa.loc[arm, '%s_delta_CI' % key],
                'stage A n': int(sa.loc[arm, '%s_n_paired' % key]),
                'this delta': d_b,
                'this CI': here.loc[arm, '%s_delta_CI' % key],
                'this n': int(here.loc[arm, 'n_seeds']),
                'shrinkage': d_b / d_a if d_a else np.nan,
            })

    rep = pd.DataFrame(rows)
    if rep.empty:
        print('Not enough arms finished yet for the comparison.')
    else:
        rep.to_csv(ABL_ROOT / 'vs_stage_a_seeds1to5.csv', index=False)
        num = ['stage A delta', 'stage A CI', 'this delta', 'this CI', 'shrinkage']
        display(rep.style.format('{:.4f}', subset=num, na_rep='-').hide(axis='index'))
        print('\n-> %s' % (ABL_ROOT / 'vs_stage_a_seeds1to5.csv').relative_to(REPO_ROOT))

---
## 8 - What this adds, and what it does not

**Adds.** An unbiased effect size for the only two relations that moved the metric in Stage A, at n=10
on seeds that played no part in selecting them, with per-seed sign consistency. If `drop_disease_protein_full`
loses on 10/10 fresh seeds with a CI well clear of zero, that is the headline number to quote - not
Stage A's -0.1042, which was selected out of 22 arms on the same five seeds that measured it.

**Does not add.** This is still the combined graph-and-signature ablation, so it cannot attribute the
effect to message passing or to metric learning. That split is
`ablation_signature_only.ipynb`, which leaves both relations in the graph and removes them only from
`disease_etypes`; and `ablation_encoder_swap.ipynb`, which holds the KG and the signature fixed while
replacing the encoder. The three together are what support the claim that the KG reaches the prediction
through the similarity signature rather than through the GNN.

**Pooling note.** Do not concatenate these ten seeds with Stage A's five into a single n=15 mean for
these two arms while every other Stage A arm stays at n=5 - the summary would mix seed sets per arm and
the cross-arm ranking would stop being comparable. Keep the two grids separate and report the
replication table in 7.1. If a pooled estimate is wanted, pool deliberately and say so, and note that
seeds 1-5 are the selection set.

**Seed overlap.** Seeds 10-19 are disjoint from Stage A's 1-5, but seed 10 is also used by
`ablation_signature_only.ipynb` and `ablation_encoder_swap.ipynb` (both run 1-10). That is harmless -
split seeds index a cached split, so the same seed means the same split everywhere - and it is why the
baseline import can sometimes reuse a run for seed 10.